# Cora HMRW Analysis

Load Cora hypergraph data and predefined splits for experiments.

In [23]:
from pathlib import Path
import pickle
import numpy as np

cora_dir = Path('/Users/lusineadunts/Documents/omms/Hilary/dissertation/data/trad_data/cocitation/cora')
if not cora_dir.exists():
    raise FileNotFoundError(f'Cora folder not found at: {cora_dir}')

with open(cora_dir / 'features.pickle', 'rb') as f:
    features = pickle.load(f)
with open(cora_dir / 'labels.pickle', 'rb') as f:
    labels = np.asarray(pickle.load(f))
with open(cora_dir / 'hypergraph.pickle', 'rb') as f:
    hypergraph_raw = pickle.load(f)

/var/folders/8z/x99j21nn30vcxmd912tgx33r0000gp/T/ipykernel_59423/659859545.py:10: DeprecationWarning: Please import `csr_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.csr` namespace is deprecated and will be removed in SciPy 2.0.0.
  features = pickle.load(f)


In [ ]:
# Load all predefined split files
splits_dir = cora_dir / 'splits'
split_files = sorted([p for p in splits_dir.glob('*.pickle')])

splits = []
for p in split_files:
    with open(p, 'rb') as f:
        splits.append(pickle.load(f))

Loaded Cora: n_nodes=2708, n_features=1433, n_edges=1579
Loaded 10 split(s)


In [57]:
len(splits[0]["train"]), len(splits[0]["test"])

(140, 2568)

In [ ]:
# 2708 nodes

2708

In [50]:
nodes = []
for edge in hypergraph_raw.values():
    for node in edge:
        if node not in nodes:
            nodes.append(node)

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


def evaluate_embeddings(embeddings, labels, splits):
    """Simple split-based evaluation with logistic regression."""
    labels = np.asarray(labels)
    rows = []

    for i, split in enumerate(splits):
        train_idx = np.asarray(split["train_idx"]).astype(int)
        test_idx = np.asarray(split["test_idx"]).astype(int)

        clf = LogisticRegression(max_iter=3000)
        clf.fit(embeddings[train_idx], labels[train_idx])
        pred = clf.predict(embeddings[test_idx])

        rows.append(
            {
                "split": i,
                "micro_f1": f1_score(labels[test_idx], pred, average="micro"),
                "macro_f1": f1_score(labels[test_idx], pred, average="macro"),
            }
        )

    per_split = pd.DataFrame(rows)
    summary = per_split[["micro_f1", "macro_f1"]].agg(["mean", "std"])
    return per_split, summary

In [ ]:
import matplotlib.pyplot as plt
from hypergraphx import Hypergraph
from src.utils import hmrw_embeddings

# Build Hypergraphx object from pickled Cora hypergraph
# Expected format: {edge_id: set/list of node ids}
edge_list = [tuple(v) for v in hypergraph_raw.values() if len(v) > 0]

# hypergraph_raw: {node_id: {edge_id1, edge_id2, ...}}
# edge_to_nodes = {}
# for node, edge_ids in hypergraph_raw.items():
#     for e in edge_ids:
#         edge_to_nodes.setdefault(e, set()).add(node)

# edge_list = [tuple(nodes) for nodes in edge_to_nodes.values() if len(nodes) > 0]
H = Hypergraph(edge_list)
print(len(H.get_nodes()), len(H.get_edges()))

# H = Hypergraph(edge_list)
# print(f'Constructed Hypergraph: n_nodes={len(H.get_nodes())}, n_edges={len(H.get_edges())}')

1579 1328


In [28]:
# Sweep p_edge (keep everything else fixed)
p_edge_values = [0.5, 1.0, 2.0, 4.0]

fixed = dict(
    p=1.0,
    q=1.0,
    q_edge=1.0,
    num_walks=10,      # walks per node
    walk_length=10,
    window=5,
    dim=64,
    workers=1,
)

rows = []
for p_edge in p_edge_values:
    embeddings = hmrw_embeddings(H, p_edge=p_edge, seed=42, **fixed)
    per_split, summary = evaluate_embeddings(embeddings, labels, splits)
    rows.append(
        {
            "p_edge": p_edge,
            "micro_f1_mean": float(summary.loc["mean", "micro_f1"]),
            "micro_f1_std": float(summary.loc["std", "micro_f1"]),
            "macro_f1_mean": float(summary.loc["mean", "macro_f1"]),
            "macro_f1_std": float(summary.loc["std", "macro_f1"]),
        }
    )

pedge_results = pd.DataFrame(rows).sort_values("p_edge")
pedge_results

NameError: name 'evaluate_embeddings' is not defined

In [ ]:
# Line plots
plt.style.use("ggplot")
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)

axes[0].plot(pedge_results["p_edge"], pedge_results["micro_f1_mean"], marker="o", color="tab:red")
axes[0].set_xlabel("p_edge")
axes[0].set_ylabel("Micro-F1")
axes[0].set_title("Micro-F1 vs p_edge")

axes[1].plot(pedge_results["p_edge"], pedge_results["macro_f1_mean"], marker="o", color="tab:red")
axes[1].set_xlabel("p_edge")
axes[1].set_ylabel("Macro-F1")
axes[1].set_title("Macro-F1 vs p_edge")

for ax in axes:
    ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.show()